# 🤖 Chatbot IHC — DSPy + Ollama + SQLite + Telegram
**Fluxo:** Pergunta do usuário → DSPy (Ollama/gemma3:1b) gera SQL → SQLite executa → Resposta no Telegram

> 💡 **Sem custo de API!** Roda 100% local via Ollama — o Colab expõe o Ollama via ngrok.

---
## Passo 1: Instalar dependências

In [ ]:
!pip install dspy-ai python-telegram-bot requests pyngrok --quiet

## Passo 2: Configurar credenciais
> ⚠️ Apenas o token do Telegram é necessário. O Ollama roda localmente (sem custo).
>
> Para usar no **Google Colab**, você precisará do ngrok para expor o Ollama. Obtenha um token gratuito em https://ngrok.com

In [ ]:
TELEGRAM_TOKEN = "SEU_TOKEN_AQUI"  # @BotFather no Telegram
NGROK_TOKEN     = "SEU_TOKEN_NGROK" # https://ngrok.com (gratuito)
OLLAMA_MODEL    = "gemma3:1b"        # modelo a usar (precisa estar instalado)

## Passo 3: Criar e popular o banco SQLite

In [ ]:
import sqlite3

DB_PATH = "company.db"

def create_database():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Tabela de departamentos
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS departments (
            id      INTEGER PRIMARY KEY AUTOINCREMENT,
            name    TEXT NOT NULL,
            budget  REAL NOT NULL
        )
    """)

    # Tabela de funcionários
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS employees (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            name       TEXT NOT NULL,
            department TEXT NOT NULL,
            salary     REAL NOT NULL,
            hire_date  TEXT NOT NULL
        )
    """)

    # Tabela de projetos
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS projects (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            name       TEXT NOT NULL,
            budget     REAL NOT NULL,
            status     TEXT NOT NULL,
            department TEXT NOT NULL
        )
    """)

    # Dados de departamentos
    departments = [
        ("Engineering",  500000),
        ("Marketing",    300000),
        ("HR",           150000),
        ("Finance",      250000),
        ("Sales",        400000),
    ]
    cursor.executemany(
        "INSERT OR IGNORE INTO departments (name, budget) VALUES (?, ?)",
        departments
    )

    # Dados de funcionários
    employees = [
        ("Alice Silva",    "Engineering", 95000, "2020-01-15"),
        ("Bob Santos",     "Engineering", 88000, "2019-03-20"),
        ("Carol Oliveira", "Engineering", 102000, "2018-07-01"),
        ("David Lima",     "Marketing",   75000, "2021-05-10"),
        ("Eva Costa",      "Marketing",   82000, "2020-11-25"),
        ("Frank Rocha",    "HR",           65000, "2022-02-14"),
        ("Grace Mendes",   "Finance",      91000, "2019-09-30"),
        ("Henry Dias",     "Sales",        78000, "2021-08-05"),
        ("Iris Ferreira",  "Sales",        84000, "2020-04-18"),
        ("João Alves",     "Engineering", 110000, "2017-12-01"),
    ]
    cursor.executemany(
        "INSERT OR IGNORE INTO employees (name, department, salary, hire_date) VALUES (?, ?, ?, ?)",
        employees
    )

    # Dados de projetos
    projects = [
        ("Sistema ERP",        250000, "In Progress",  "Engineering"),
        ("App Mobile",         180000, "In Progress",  "Engineering"),
        ("Campanha Q4",         50000, "Completed",    "Marketing"),
        ("Portal RH",           30000, "Planning",     "HR"),
        ("Relatório Fiscal",    20000, "In Progress",  "Finance"),
        ("CRM Vendas",         120000, "Planning",     "Sales"),
    ]
    cursor.executemany(
        "INSERT OR IGNORE INTO projects (name, budget, status, department) VALUES (?, ?, ?, ?)",
        projects
    )

    conn.commit()
    conn.close()
    print("✅ Banco de dados criado e populado com sucesso!")

create_database()

## Passo 4: Instalar e configurar Ollama

In [ ]:
import subprocess, time, requests, dspy
from pyngrok import ngrok, conf

# ── 1. Instalar o Ollama no Colab ─────────────────────────
print("📦 Instalando Ollama...")
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True, capture_output=True
)

# ── 2. Iniciar o servidor Ollama em background ────────────
print("🚀 Iniciando servidor Ollama...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)  # aguarda o servidor subir

# ── 3. Verificar se está rodando ──────────────────────────
def setup_ollama():
    """Configure DSPy to use Ollama."""
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("✅ Ollama is running")
        return True
    except Exception:
        print("❌ Ollama not running. Start with: ollama serve")
        return False

if not setup_ollama():
    raise RuntimeError("Ollama não iniciou. Verifique a instalação.")

# ── 4. Baixar o modelo gemma3:1b ──────────────────────────
print(f"⬇️  Baixando modelo {OLLAMA_MODEL} (pode demorar na primeira vez)...")
result = subprocess.run(
    ["ollama", "pull", OLLAMA_MODEL],
    capture_output=True, text=True
)
print(result.stdout or "Modelo pronto!")

# ── 5. Expor via ngrok para o Colab conseguir acessar ─────
print("🌐 Configurando ngrok...")
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(11434, "http")
OLLAMA_BASE_URL = tunnel.public_url
print(f"✅ Ollama exposto em: {OLLAMA_BASE_URL}")

# ── 6. Configurar DSPy com Ollama ─────────────────────────
dspy.settings.configure(
    lm=dspy.LM(
        model=f"ollama/{OLLAMA_MODEL}",
        api_base="http://localhost:11434",
        max_tokens=500,
        temperature=0.7,
    )
)

print(f"✅ DSPy configurado com Ollama ({OLLAMA_MODEL})")

## Passo 5: Exemplos de treino (few-shot)

In [ ]:
def create_examples():
    """Training examples for few-shot learning."""
    return [
        dspy.Example(
            question="How many employees are in the Engineering department?",
            sql_query="SELECT COUNT(*) FROM employees WHERE department = 'Engineering'"
        ).with_inputs("question"),

        dspy.Example(
            question="What is the highest salary in Engineering?",
            sql_query="SELECT MAX(salary) FROM employees WHERE department = 'Engineering'"
        ).with_inputs("question"),

        dspy.Example(
            question="List all employees with salary above 90000",
            sql_query="SELECT name, salary FROM employees WHERE salary > 90000"
        ).with_inputs("question"),

        dspy.Example(
            question="Which department has the highest budget?",
            sql_query="SELECT name, budget FROM departments ORDER BY budget DESC LIMIT 1"
        ).with_inputs("question"),

        dspy.Example(
            question="Show all projects in progress",
            sql_query="SELECT name, budget, status FROM projects WHERE status = 'In Progress'"
        ).with_inputs("question"),
    ]

examples = create_examples()
print(f"✅ {len(examples)} exemplos de treino carregados")

## Passo 6: Definir o módulo DSPy (Text-to-SQL)

In [ ]:
# Schema do banco para contexto do LLM
DB_SCHEMA = """
Tables:
- employees(id, name, department, salary, hire_date)
- departments(id, name, budget)
- projects(id, name, budget, status, department)

Status values for projects: 'In Progress', 'Completed', 'Planning'
Departments: Engineering, Marketing, HR, Finance, Sales
"""

class TextToSQL(dspy.Signature):
    """Convert a natural language question to a valid SQLite SQL query."""
    schema:   str = dspy.InputField(desc="Database schema with table and column names")
    question: str = dspy.InputField(desc="Natural language question about the database")
    sql_query: str = dspy.OutputField(desc="Valid SQLite SQL query. Return ONLY the SQL, no explanation.")


class SQLChatbot(dspy.Module):
    def __init__(self, examples):
        super().__init__()
        self.predict = dspy.Predict(TextToSQL)
        self.examples = examples

    def forward(self, question):
        result = self.predict(schema=DB_SCHEMA, question=question)
        return result


chatbot_module = SQLChatbot(examples)
print("✅ Módulo DSPy criado")

## Passo 7: Treinar com BootstrapFewShot

In [ ]:
from dspy.teleprompt import BootstrapFewShot

def sql_metric(example, pred, trace=None):
    """Verifica se a query gerada é SQL válida e executável."""
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute(pred.sql_query)
        conn.close()
        return True
    except Exception:
        return False

# Treina com few-shot bootstrapping
optimizer = BootstrapFewShot(metric=sql_metric, max_bootstrapped_demos=3)
trained_chatbot = optimizer.compile(chatbot_module, trainset=examples)

print("✅ Modelo treinado com BootstrapFewShot!")

## Passo 8: Funções auxiliares — executar SQL e formatar resposta

In [ ]:
import re

def clean_sql(raw: str) -> str:
    """Remove blocos markdown e espaços extras do SQL gerado."""
    raw = re.sub(r"```[\w]*", "", raw).strip()
    return raw


def execute_query(sql: str) -> str:
    """Executa a query SQLite e retorna resultado formatado."""
    try:
        sql = clean_sql(sql)
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(sql)
        rows = cursor.fetchall()
        col_names = [desc[0] for desc in cursor.description] if cursor.description else []
        conn.close()

        if not rows:
            return "Nenhum resultado encontrado."

        # Formata a tabela de resultado
        lines = [" | ".join(col_names)] if col_names else []
        lines.append("-" * max(len(lines[0]) if lines else 10, 10))
        for row in rows:
            lines.append(" | ".join(str(v) for v in row))

        return "\n".join(lines)

    except Exception as e:
        return f"❌ Erro ao executar query: {e}"


def process_question(question: str) -> str:
    """Pipeline completo: pergunta → SQL → resultado."""
    try:
        result = trained_chatbot(question=question)
        sql = clean_sql(result.sql_query)
        answer = execute_query(sql)
        return f"🔍 *Query gerada:*\n`{sql}`\n\n📊 *Resultado:*\n```\n{answer}\n```"
    except Exception as e:
        return f"❌ Erro ao processar sua pergunta: {e}"


# Teste rápido
print(process_question("How many employees are in the Engineering department?"))

## Passo 9: Bot do Telegram

In [ ]:
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, ContextTypes, filters

# ── Handlers ──────────────────────────────────────────────

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    msg = (
        "👋 Olá! Sou o *SQL Chatbot IHC*.\n\n"
        "Faça perguntas em inglês sobre o banco de dados da empresa.\n\n"
        "*Exemplos:*\n"
        "• How many employees are in Engineering?\n"
        "• Which department has the highest budget?\n"
        "• List employees with salary above 90000\n"
        "• Show all projects in progress\n\n"
        "Use /help para ver mais exemplos."
    )
    await update.message.reply_text(msg, parse_mode="Markdown")


async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    msg = (
        "📖 *Perguntas que você pode fazer:*\n\n"
        "*Funcionários:*\n"
        "• How many employees are in [department]?\n"
        "• What is the highest salary in [department]?\n"
        "• List all employees with salary above [value]\n\n"
        "*Departamentos:*\n"
        "• Which department has the highest budget?\n"
        "• Show all departments\n\n"
        "*Projetos:*\n"
        "• Show all projects in progress\n"
        "• How many projects are completed?\n"
    )
    await update.message.reply_text(msg, parse_mode="Markdown")


async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    question = update.message.text
    await update.message.reply_text("⏳ Processando sua pergunta...")

    response = process_question(question)
    await update.message.reply_text(response, parse_mode="Markdown")


# ── Iniciar o bot ─────────────────────────────────────────

print("🚀 Iniciando bot do Telegram...")
print("   Pressione Ctrl+C para parar.\n")

app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(CommandHandler("help",  help_command))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

app.run_polling()